In [ ]:
# inlegalbert_bilstm_mha_crf_rrc_v2_weighted.py
#
# Architecture:
#   InLegalBERT  →  BiLSTM  →  Multi-Head Attention Pooling  →  Linear  →  CRF
#
# CHANGES vs v2:
#   [NEW] Class-weighted cross-entropy for imbalance handling.
#         Four modifications total:
#         1. compute_class_weights()  — new helper function (after detect_rare_classes).
#         2. Model.__init__()         — ce_loss now accepts an optional weight tensor.
#         3. Model.set_ce_weights()   — method to inject weights after construction.
#         4. main()                   — computes weights from train_docs and injects them.
#
# FIXES vs original weighted version:
#   [FIX-1]  AUX_CE_WEIGHT reduced 0.2 → 0.05.
#   [FIX-2]  CE_WEIGHT_CAP reduced 10.0 → 5.0.
#   [FIX-3]  CE_WEIGHT_STRATEGY changed "inv_freq" → "sqrt_inv".
#   [FIX-4]  Normalisation in compute_class_weights() changed mean → max.
#
# RESUME / CHECKPOINT CHANGES  [CKPT]:
#   [CKPT-1]  CHECKPOINT_DIR constant — dedicated folder for resume state.
#   [CKPT-2]  save_checkpoint()       — saves full resume state after every epoch:
#               • model state_dict
#               • optimizer state_dict
#               • scheduler state_dict
#               • epoch number
#               • best_f1 so far
#               • EarlyStopping counter + best_score
#               • complete history rows list
#             Only ONE file is kept (last_checkpoint.pt) to save disk space.
#             The best model is still saved separately in BEST_MODEL_DIR.
#   [CKPT-3]  load_checkpoint()       — restores all of the above; returns
#               resume_epoch so the training loop starts from epoch+1.
#   [CKPT-4]  Trainer.train()         — calls save_checkpoint() at the end of
#               every epoch; calls load_checkpoint() at the start if a checkpoint
#               file exists; prints a clear RESUMING message showing which epoch
#               training is continuing from.
#   [CKPT-5]  main()                  — creates CHECKPOINT_DIR; nothing else changes.
#

import os, json, random, time
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup,
)
from torchcrf import CRF
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score,
    precision_score, recall_score,
)

# ═══════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════
INLEGALBERT_MODEL_NAME = "law-ai/InLegalBERT"
TRAIN_PATH = "dataset/build_train.jsonl"
DEV_PATH   = "dataset/build_dev.jsonl"
TEST_PATH  = "dataset/build_test.jsonl"
OUT_DIR    = "rrc_bilstm_mha_crf_v2_weighted_logs"
BEST_MODEL_DIR = os.path.join(OUT_DIR, "best_model")

# [CKPT-1]  Dedicated folder for the resume checkpoint.
#           Kept separate from BEST_MODEL_DIR so a crash during saving
#           cannot corrupt the best model weights.
CHECKPOINT_DIR = os.path.join(OUT_DIR, "checkpoints")
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "last_checkpoint.pt")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)   # [CKPT-5]

SEED            = 42
MAX_SEQ_LENGTH  = 32
BATCH_DOCS      = 2
NUM_EPOCHS      = 60
BERT_LR         = 1e-5
HEAD_LR         = 5e-4
WEIGHT_DECAY    = 0.05
GRAD_CLIP       = 1.0
DROPOUT         = 0.4

BERT_FREEZE_LAYERS  = 8
BERT_LR_DECAY       = 0.9

SENT_LSTM_HIDDEN = 128
SENT_LSTM_LAYERS = 2

MHA_HEADS        = 4
MHA_DROPOUT      = 0.1

CTX_LSTM_HIDDEN  = 64
CTX_LSTM_LAYERS  = 2

AUX_CE_WEIGHT    = 0.05          # [FIX-1]
LABEL_SMOOTHING  = 0.1

CE_WEIGHT_STRATEGY = "sqrt_inv"  # [FIX-3]
CE_WEIGHT_CAP      = 5.0         # [FIX-2]

ES_PATIENCE      = 10
ES_MIN_DELTA     = 1e-4

WARMUP_RATIO    = 0.05
GRADIENT_ACCUMULATION_STEPS = 2
RARE_THRESHOLD  = 0.05

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id   = {lbl: i for i, lbl in enumerate(LABELS)}
id2label   = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
DEVICE     = "cuda:0" if torch.cuda.is_available() else "cpu"


# ═══════════════════════════════════════════════════════════
# SEED
# ═══════════════════════════════════════════════════════════
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


# ═══════════════════════════════════════════════════════════
# CHECKPOINT HELPERS  [CKPT-2 & CKPT-3]
# ═══════════════════════════════════════════════════════════
def save_checkpoint(
    epoch:         int,
    model:         nn.Module,
    optimizer:     torch.optim.Optimizer,
    scheduler,
    best_f1:       float,
    best_state:    dict,
    early_stopper,
    history:       list,
    path:          str = CHECKPOINT_PATH,
):
    """
    [CKPT-2]  Save a complete resume snapshot to `path`.

    Contents
    --------
    epoch         : last completed epoch number (1-based)
    model_state   : model.state_dict()  — all weights on CPU
    optimizer_state : optimizer.state_dict()
    scheduler_state : scheduler.state_dict()
    best_f1       : highest val_macro_f1 seen so far
    best_state    : model state_dict at the best epoch (CPU)
    es_counter    : EarlyStopping.counter
    es_best_score : EarlyStopping.best_score
    history       : list of per-epoch metric dicts (for CSV / plots)

    Only one file is kept — it is overwritten after every epoch so disk
    usage stays constant regardless of how long training runs.
    """
    # Move model weights to CPU before pickling to avoid device mismatches
    # on resume (e.g. if the job resumes on a different GPU index).
    cpu_model_state = {k: v.cpu() for k, v in model.state_dict().items()}
    cpu_best_state  = {k: v.cpu() for k, v in best_state.items()} \
                      if best_state is not None else None

    torch.save(
        {
            "epoch":           epoch,
            "model_state":     cpu_model_state,
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "best_f1":         best_f1,
            "best_state":      cpu_best_state,
            "es_counter":      early_stopper.counter,
            "es_best_score":   early_stopper.best_score,
            "history":         history,
        },
        path,
    )


def load_checkpoint(
    path:      str,
    model:     nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler,
    early_stopper,
    device:    str = DEVICE,
):
    """
    [CKPT-3]  Restore training state from `path`.

    Returns
    -------
    resume_epoch : int  — the epoch to START from (last_epoch + 1)
    best_f1      : float
    best_state   : dict | None  — model state_dict at best epoch (CPU)
    history      : list of per-epoch metric dicts

    Side effects
    ------------
    Loads weights into `model`, restores `optimizer` and `scheduler`
    state dicts (mapped to `device`), and restores `early_stopper`
    counters so ES patience is preserved exactly.
    """
    ckpt = torch.load(path, map_location=device)

    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])

    early_stopper.counter    = ckpt["es_counter"]
    early_stopper.best_score = ckpt["es_best_score"]

    resume_epoch = ckpt["epoch"] + 1        # next epoch to run
    best_f1      = ckpt["best_f1"]
    best_state   = ckpt["best_state"]
    history      = ckpt["history"]

    return resume_epoch, best_f1, best_state, history


# ═══════════════════════════════════════════════════════════
# EARLY STOPPING
# ═══════════════════════════════════════════════════════════
class EarlyStopping:
    def __init__(self, patience=ES_PATIENCE, min_delta=ES_MIN_DELTA):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = -1.0
        self.counter    = 0
        self.stop       = False

    def step(self, score: float) -> bool:
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


# ═══════════════════════════════════════════════════════════
# PARAMETER COUNTER
# ═══════════════════════════════════════════════════════════
def count_parameters(model):
    component_map = {
        "bert":              "InLegalBERT Encoder",
        "sent_bilstm":       "Sentence BiLSTM",
        "mha_pooling":       "Multi-Head Attn Pooling",
        "ctx_bilstm":        "Context BiLSTM",
        "classifier":        "Classifier Head",
        "crf":               "CRF",
        "dropout":           "Dropout",
    }
    rows = []
    for attr, name in component_map.items():
        module = getattr(model, attr, None)
        if module is None:
            continue
        trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
        frozen    = sum(p.numel() for p in module.parameters() if not p.requires_grad)
        rows.append({
            "Component":        name,
            "Trainable Params": trainable,
            "Frozen Params":    frozen,
            "Total Params":     trainable + frozen,
        })
    total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    rows.append({
        "Component":        "── TOTAL ──",
        "Trainable Params": total_trainable,
        "Frozen Params":    total_frozen,
        "Total Params":     total_trainable + total_frozen,
    })
    print("\n" + "=" * 70)
    print("MODEL PARAMETER SUMMARY")
    print("=" * 70)
    print(f"  {'Component':<30} {'Trainable':>14} {'Frozen':>10} {'Total':>12}")
    print("-" * 70)
    for r in rows:
        sep = "─" * 70 if r["Component"] == "── TOTAL ──" else ""
        if sep:
            print(sep)
        print(f"  {r['Component']:<30} "
              f"{r['Trainable Params']:>14,} "
              f"{r['Frozen Params']:>10,} "
              f"{r['Total Params']:>12,}")
    print("=" * 70)
    return total_trainable, total_frozen, rows


# ═══════════════════════════════════════════════════════════
# RARE-CLASS AUTO-DETECTION
# ═══════════════════════════════════════════════════════════
def detect_rare_classes(docs, threshold=RARE_THRESHOLD):
    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)
    label_freqs = {id2label[i]: counts.get(i, 0) / total for i in range(NUM_LABELS)}
    rare_labels = [id2label[i] for i in range(NUM_LABELS)
                   if label_freqs[id2label[i]] <= threshold]
    rare_ids    = [label2id[l] for l in rare_labels]

    print(f"\n📊 Label frequency analysis (threshold ≤ {threshold*100:.0f}%):")
    for lbl in LABELS:
        freq  = label_freqs[lbl]
        flag  = " ← RARE" if lbl in rare_labels else ""
        count = counts.get(label2id[lbl], 0)
        print(f"   {lbl:<20} {freq*100:5.2f}%  ({count:5d} samples){flag}")
    print(f"\n   Rare classes ({len(rare_labels)}): {rare_labels}\n")
    return rare_labels, rare_ids, label_freqs


# ═══════════════════════════════════════════════════════════
# CLASS-WEIGHT COMPUTATION
# ═══════════════════════════════════════════════════════════
def compute_class_weights(
    docs,
    strategy: str  = CE_WEIGHT_STRATEGY,
    cap:      float = CE_WEIGHT_CAP,
    device:   str   = DEVICE,
) -> torch.Tensor:
    """
    Compute a per-class weight tensor from training document label counts.

    Strategies
    ----------
    "inv_freq"  : w_i = N_total / (N_classes * count_i)
    "sqrt_inv"  : w_i = sqrt(N_total / count_i)   [FIX-3: default]
    "none"      : uniform weights (disables weighting)

    [FIX-4]  Normalised by max() so all weights ∈ (0, 1].
    [FIX-2]  cap=5.0 prevents near-absent classes from dominating.
    """
    if strategy == "none":
        weights = torch.ones(NUM_LABELS, dtype=torch.float, device=device)
        print("\n⚖️  CE class weights: uniform (strategy='none')\n")
        return weights

    all_ids = [lid for _, labs in docs for lid in labs]
    total   = len(all_ids)
    counts  = Counter(all_ids)

    raw = torch.zeros(NUM_LABELS, dtype=torch.float)
    for i in range(NUM_LABELS):
        c = counts.get(i, 0)
        if c == 0:
            raw[i] = cap
        elif strategy == "inv_freq":
            raw[i] = total / (NUM_LABELS * c)
        elif strategy == "sqrt_inv":
            raw[i] = (total / c) ** 0.5
        else:
            raise ValueError(f"Unknown CE_WEIGHT_STRATEGY: {strategy!r}")

    raw = torch.clamp(raw, max=cap)
    raw = raw / raw.max()              # [FIX-4]

    print(f"\n⚖️  CE class weights (strategy='{strategy}', cap={cap}, norm=max):")
    print(f"   {'Label':<20} {'Count':>7} {'Weight':>8}")
    print(f"   {'-'*38}")
    for i in range(NUM_LABELS):
        lbl = id2label[i]
        cnt = counts.get(i, 0)
        print(f"   {lbl:<20} {cnt:>7d} {raw[i].item():>8.4f}")
    print()

    return raw.to(device)


# ═══════════════════════════════════════════════════════════
# DATA LOADING
# ═══════════════════════════════════════════════════════════
def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data


def extract_docs(docs, max_sents=256):
    all_docs = []
    for doc in docs:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs  = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val  = item.get("value", {})
                    text = val.get("text", "").strip()
                    lbl  = val.get("labels", ["NONE"])[0]
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(lbl, label2id["NONE"]))
        if not sents or len(sents) != len(labs):
            continue
        sents = sents[:max_sents]
        labs  = labs[:max_sents]
        all_docs.append((sents, labs))
    return all_docs


# ═══════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════
class RRCDataset(Dataset):
    def __init__(self, docs, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs       = docs
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        sents, labels = self.docs[idx]
        enc = self.tokenizer(
            sents,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc.get(
                "token_type_ids",
                torch.zeros_like(enc["input_ids"])
            ),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def collate_rrc(batch):
    T_max = max(b["input_ids"].shape[0] for b in batch)
    L     = batch[0]["input_ids"].shape[1]
    B     = len(batch)
    input_ids      = torch.zeros(B, T_max, L, dtype=torch.long)
    attention_mask = torch.zeros(B, T_max, L, dtype=torch.long)
    token_type_ids = torch.zeros(B, T_max, L, dtype=torch.long)
    labels         = torch.full((B, T_max), fill_value=-100, dtype=torch.long)
    lengths        = torch.zeros(B, dtype=torch.long)
    for i, b in enumerate(batch):
        t = b["input_ids"].shape[0]
        input_ids[i, :t]      = b["input_ids"]
        attention_mask[i, :t] = b["attention_mask"]
        token_type_ids[i, :t] = b["token_type_ids"]
        labels[i, :t]         = b["labels"]
        lengths[i]            = t
    return input_ids, attention_mask, token_type_ids, labels, lengths


# ═══════════════════════════════════════════════════════════
# MULTI-HEAD ATTENTION POOLING
# ═══════════════════════════════════════════════════════════
class MultiHeadAttentionPooling(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads
        self.query = nn.Parameter(torch.empty(1, num_heads, 1, self.head_dim))
        nn.init.trunc_normal_(self.query, std=0.02)
        self.key_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.val_proj  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, hidden_dim, bias=True)
        self.attn_drop = nn.Dropout(dropout)
        self.scale     = self.head_dim ** -0.5

    def forward(self, x: torch.Tensor,
                key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        N, L, H = x.shape
        K = self.key_proj(x)
        V = self.val_proj(x)
        K = K.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(N, L, self.num_heads, self.head_dim).transpose(1, 2)
        Q = self.query.expand(N, -1, -1, -1)
        attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        if key_padding_mask is not None:
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            attn_weights = attn_weights.masked_fill(mask, -1e9)
        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_weights = self.attn_drop(attn_weights)
        context = torch.matmul(attn_weights, V).squeeze(2).reshape(N, H)
        return self.out_proj(context)


# ═══════════════════════════════════════════════════════════
# MODEL
# ═══════════════════════════════════════════════════════════
class InLegalBERT_BiLSTM_MHA_CRF(nn.Module):

    def __init__(
        self,
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
        class_weights    = None,
    ):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(bert_model_name)
        self.bert_dim = self.bert.config.hidden_size
        self.dropout  = nn.Dropout(dropout)
        self._freeze_bert_layers(freeze_layers)

        self.sent_bilstm = nn.LSTM(
            input_size    = self.bert_dim,
            hidden_size   = sent_lstm_hidden,
            num_layers    = sent_lstm_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if sent_lstm_layers > 1 else 0.0,
        )
        sent_out_dim = sent_lstm_hidden * 2

        self.mha_pooling = MultiHeadAttentionPooling(
            hidden_dim = sent_out_dim,
            num_heads  = mha_heads,
            dropout    = mha_dropout,
        )
        self.sent_layer_norm = nn.LayerNorm(sent_out_dim)

        self.ctx_bilstm = nn.LSTM(
            input_size    = sent_out_dim,
            hidden_size   = ctx_lstm_hidden,
            num_layers    = ctx_lstm_layers,
            bidirectional = True,
            batch_first   = True,
            dropout       = dropout if ctx_lstm_layers > 1 else 0.0,
        )
        ctx_out_dim = ctx_lstm_hidden * 2

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(ctx_out_dim, ctx_out_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ctx_out_dim // 2, num_labels),
        )
        self.crf = CRF(num_tags=num_labels, batch_first=True)

        self.ce_loss = nn.CrossEntropyLoss(
            weight          = class_weights,
            label_smoothing = LABEL_SMOOTHING,
            ignore_index    = -100,
            reduction       = "mean",
        )

    def set_ce_weights(self, class_weights: torch.Tensor):
        self.ce_loss = nn.CrossEntropyLoss(
            weight          = class_weights,
            label_smoothing = LABEL_SMOOTHING,
            ignore_index    = -100,
            reduction       = "mean",
        )

    def _freeze_bert_layers(self, n_freeze: int):
        for param in self.bert.embeddings.parameters():
            param.requires_grad = False
        encoder_layers = self.bert.encoder.layer
        for i in range(min(n_freeze, len(encoder_layers))):
            for param in encoder_layers[i].parameters():
                param.requires_grad = False
        n_total   = len(encoder_layers)
        n_trained = n_total - n_freeze
        print(f"\n❄️  BERT layers frozen: embeddings + layers 0-{n_freeze-1}.")
        print(f"🔥 BERT layers trainable: layers {n_freeze}-{n_total-1} "
              f"({n_trained} layers) + pooler.\n")

    def encode_sentences(self, input_ids, attention_mask, token_type_ids,
                         lengths=None):
        B, T, L = input_ids.shape
        N = B * T
        flat_ids   = input_ids.view(N, L)
        flat_mask  = attention_mask.view(N, L)
        flat_types = token_type_ids.view(N, L)
        valid = flat_mask.sum(dim=-1) > 0
        token_embs_all = flat_ids.new_zeros(N, L, self.bert_dim, dtype=torch.float)
        if valid.any():
            out = self.bert(
                input_ids      = flat_ids[valid],
                attention_mask = flat_mask[valid],
                token_type_ids = flat_types[valid],
            )
            token_embs_all[valid] = out.last_hidden_state.to(token_embs_all.dtype)
        token_embs_all = self.dropout(token_embs_all)
        lstm_out, _ = self.sent_bilstm(token_embs_all)
        lstm_out    = self.dropout(lstm_out)
        pad_mask = (flat_mask == 0).clone()
        pad_mask[~valid] = False
        sent_vecs = self.mha_pooling(lstm_out, key_padding_mask=pad_mask)
        sent_vecs = self.sent_layer_norm(sent_vecs)
        sent_vecs = sent_vecs * valid.unsqueeze(-1).to(sent_vecs.dtype)
        return sent_vecs.view(B, T, -1)

    def forward(self, input_ids, attention_mask, token_type_ids,
                labels=None, lengths=None):
        sent_vecs = self.encode_sentences(
            input_ids, attention_mask, token_type_ids, lengths=lengths
        )
        sent_vecs = self.dropout(sent_vecs)

        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                sent_vecs, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_out, _ = self.ctx_bilstm(packed)
            ctx_out, _    = nn.utils.rnn.pad_packed_sequence(
                packed_out, batch_first=True
            )
        else:
            ctx_out, _ = self.ctx_bilstm(sent_vecs)

        ctx_out   = self.dropout(ctx_out)
        emissions = self.classifier(ctx_out)
        emissions = torch.nan_to_num(emissions, nan=0.0, posinf=1e4, neginf=-1e4)

        if lengths is not None:
            B, T, _ = emissions.shape
            mask = torch.zeros(B, T, dtype=torch.bool, device=emissions.device)
            for i, l in enumerate(lengths):
                mask[i, :l] = True
        elif labels is not None:
            mask = (labels != -100)
        else:
            mask = torch.ones(emissions.shape[:2], dtype=torch.bool,
                              device=emissions.device)

        if labels is not None:
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0
            crf_loss = -self.crf(emissions, safe_labels, mask=mask, reduction="mean")
            B2, T2, C = emissions.shape
            ce_loss = self.ce_loss(
                emissions.reshape(B2 * T2, C),
                labels.reshape(B2 * T2),
            )
            loss = crf_loss + AUX_CE_WEIGHT * ce_loss
            return loss, emissions
        else:
            return self.crf.decode(emissions, mask=mask), emissions


# ═══════════════════════════════════════════════════════════
# METRICS HELPER
# ═══════════════════════════════════════════════════════════
def compute_all_metrics(all_trues, all_preds, rare_ids, split_name=""):
    str_trues = [id2label[x] for x in all_trues]
    str_preds = [id2label[x] for x in all_preds]
    macro_f1    = f1_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_f1    = f1_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_f1 = f1_score(all_trues, all_preds, average="weighted", zero_division=0)
    macro_prec    = precision_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_prec    = precision_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_prec = precision_score(all_trues, all_preds, average="weighted", zero_division=0)
    macro_rec    = recall_score(all_trues, all_preds, average="macro",    zero_division=0)
    micro_rec    = recall_score(all_trues, all_preds, average="micro",    zero_division=0)
    weighted_rec = recall_score(all_trues, all_preds, average="weighted", zero_division=0)
    acc = accuracy_score(all_trues, all_preds)
    per_class_f1   = f1_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                              average=None, zero_division=0)
    per_class_prec = precision_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                     average=None, zero_division=0)
    per_class_rec  = recall_score(all_trues, all_preds, labels=list(range(NUM_LABELS)),
                                  average=None, zero_division=0)
    per_class_metrics = {
        id2label[i]: {
            "f1":        float(per_class_f1[i]),
            "precision": float(per_class_prec[i]),
            "recall":    float(per_class_rec[i]),
        }
        for i in range(NUM_LABELS)
    }
    present_rare = [r for r in rare_ids if r in all_trues]
    if present_rare:
        rare_f1   = f1_score(all_trues, all_preds, labels=present_rare,
                             average="macro", zero_division=0)
        rare_prec = precision_score(all_trues, all_preds, labels=present_rare,
                                    average="macro", zero_division=0)
        rare_rec  = recall_score(all_trues, all_preds, labels=present_rare,
                                 average="macro", zero_division=0)
    else:
        rare_f1 = rare_prec = rare_rec = 0.0
    cls_report = classification_report(str_trues, str_preds, labels=LABELS,
                                       digits=4, zero_division=0)
    cm = confusion_matrix(str_trues, str_preds, labels=LABELS)
    return {
        "macro_f1": macro_f1, "micro_f1": micro_f1, "weighted_f1": weighted_f1,
        "macro_precision": macro_prec, "micro_precision": micro_prec,
        "weighted_precision": weighted_prec,
        "macro_recall": macro_rec, "micro_recall": micro_rec,
        "weighted_recall": weighted_rec,
        "rare_f1": rare_f1, "rare_precision": rare_prec, "rare_recall": rare_rec,
        "per_class_metrics": per_class_metrics, "accuracy": acc,
        "cls_report": cls_report, "cm": cm,
        "all_preds": all_preds, "all_trues": all_trues,
    }


# ═══════════════════════════════════════════════════════════
# TRAINER
# ═══════════════════════════════════════════════════════════
class Trainer:
    def __init__(self, model, device=DEVICE):
        self.model  = model.to(device)
        self.device = device

    def build_optimizer(self):
        param_groups = []
        param_groups.append({
            "params":       list(self.model.bert.pooler.parameters()),
            "lr":           BERT_LR,
            "weight_decay": WEIGHT_DECAY,
        })
        encoder_layers = self.model.bert.encoder.layer
        n_layers = len(encoder_layers)
        for i in range(n_layers - 1, BERT_FREEZE_LAYERS - 1, -1):
            depth = (n_layers - 1) - i
            lr_i  = BERT_LR * (BERT_LR_DECAY ** depth)
            params = [p for p in encoder_layers[i].parameters() if p.requires_grad]
            if params:
                param_groups.append({
                    "params":       params,
                    "lr":           lr_i,
                    "weight_decay": WEIGHT_DECAY,
                })
        head_modules = [
            self.model.sent_bilstm, self.model.mha_pooling,
            self.model.sent_layer_norm, self.model.ctx_bilstm,
            self.model.classifier, self.model.crf,
        ]
        head_params = []
        for m in head_modules:
            head_params.extend(list(m.parameters()))
        param_groups.append({
            "params":       head_params,
            "lr":           HEAD_LR,
            "weight_decay": WEIGHT_DECAY,
        })
        return torch.optim.AdamW(param_groups)

    def compute_val_loss(self, dataset):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        total_loss, n = 0.0, 0
        with torch.no_grad():
            for input_ids, attention_mask, token_type_ids, labels, lengths in loader:
                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                labels         = labels.to(self.device)
                lengths        = lengths.to(self.device)
                loss, _ = self.model(input_ids, attention_mask, token_type_ids,
                                     labels=labels, lengths=lengths)
                if not torch.isnan(loss):
                    total_loss += loss.item()
                    n += 1
        return total_loss / max(1, n)

    def train(self, train_dataset, dev_dataset, rare_ids,
              tokenizer, num_epochs=NUM_EPOCHS):

        train_loader = DataLoader(train_dataset, batch_size=BATCH_DOCS,
                                  shuffle=True, collate_fn=collate_rrc)
        optimizer    = self.build_optimizer()
        total_steps  = len(train_loader) * num_epochs // GRADIENT_ACCUMULATION_STEPS
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler    = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps   = warmup_steps,
            num_training_steps = total_steps,
        )

        early_stopper = EarlyStopping()
        history       = []
        best_f1       = -1.0
        best_state    = None
        start_epoch   = 1             # may be overridden by checkpoint

        # ── [CKPT-4]  Resume from last checkpoint if it exists ───────────
        if os.path.exists(CHECKPOINT_PATH):
            print(f"\n🔄  Checkpoint found at {CHECKPOINT_PATH}")
            try:
                start_epoch, best_f1, best_state, history = load_checkpoint(
                    path          = CHECKPOINT_PATH,
                    model         = self.model,
                    optimizer     = optimizer,
                    scheduler     = scheduler,
                    early_stopper = early_stopper,
                    device        = self.device,
                )
                print(f"    RESUMING from epoch {start_epoch}  "
                      f"(best_f1={best_f1:.4f}, "
                      f"ES counter={early_stopper.counter}/{early_stopper.patience})\n")
            except Exception as exc:
                # Corrupt checkpoint — start fresh with a warning
                print(f"    ⚠️  Could not load checkpoint ({exc}). Starting fresh.\n")
                start_epoch = 1
        else:
            print("\n▶  No checkpoint found — starting training from scratch.\n")
        # ─────────────────────────────────────────────────────────────────

        total_train_start = time.time()
        actual_epochs     = start_epoch - 1   # epochs completed before this run

        for epoch in range(start_epoch, num_epochs + 1):
            actual_epochs = epoch
            self.model.train()
            running_loss, n_steps, nan_steps = 0.0, 0, 0
            epoch_start = time.time()
            optimizer.zero_grad()

            for step, (input_ids, attention_mask,
                        token_type_ids, labels, lengths) in enumerate(train_loader):
                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                labels         = labels.to(self.device)
                lengths        = lengths.to(self.device)
                loss, _ = self.model(input_ids, attention_mask, token_type_ids,
                                     labels=labels, lengths=lengths)
                if torch.isnan(loss) or torch.isinf(loss):
                    nan_steps += 1
                    optimizer.zero_grad()
                    continue
                loss = loss / GRADIENT_ACCUMULATION_STEPS
                loss.backward()
                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()
                running_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
                n_steps += 1

            if n_steps % GRADIENT_ACCUMULATION_STEPS != 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            epoch_train_time = time.time() - epoch_start
            avg_train_loss   = running_loss / max(1, n_steps)
            val_loss         = self.compute_val_loss(dev_dataset)
            val_metrics      = self.evaluate(dev_dataset, rare_ids)

            nan_info = f" [nan={nan_steps}]" if nan_steps > 0 else ""
            print(
                f"Epoch {epoch:03d}/{num_epochs} | "
                f"train: {avg_train_loss:.4f} | val: {val_loss:.4f} | "
                f"macro_f1: {val_metrics['macro_f1']:.4f} | "
                f"rare_f1: {val_metrics['rare_f1']:.4f} | "
                f"acc: {val_metrics['accuracy']:.4f} | "
                f"ES: {early_stopper.counter}/{early_stopper.patience}{nan_info}"
            )

            row = {
                "epoch":                epoch,
                "train_loss":           avg_train_loss,
                "val_loss":             val_loss,
                "val_accuracy":         val_metrics["accuracy"],
                "val_macro_f1":         val_metrics["macro_f1"],
                "val_micro_f1":         val_metrics["micro_f1"],
                "val_weighted_f1":      val_metrics["weighted_f1"],
                "val_rare_f1":          val_metrics["rare_f1"],
                "val_macro_precision":  val_metrics["macro_precision"],
                "val_micro_precision":  val_metrics["micro_precision"],
                "val_weighted_precision": val_metrics["weighted_precision"],
                "val_rare_precision":   val_metrics["rare_precision"],
                "val_macro_recall":     val_metrics["macro_recall"],
                "val_micro_recall":     val_metrics["micro_recall"],
                "val_weighted_recall":  val_metrics["weighted_recall"],
                "val_rare_recall":      val_metrics["rare_recall"],
                "epoch_train_time_s":   epoch_train_time,
                "nan_steps":            nan_steps,
                "timestamp":            datetime.utcnow().isoformat(),
            }
            history.append(row)

            if val_metrics["macro_f1"] > best_f1 + ES_MIN_DELTA:
                best_f1    = val_metrics["macro_f1"]
                best_state = {k: v.cpu().clone()
                              for k, v in self.model.state_dict().items()}
                print(f"  ✔ New best val_macro_f1={best_f1:.4f}")

            # ── [CKPT-4]  Save resume checkpoint after every epoch ────────
            # This is done BEFORE the early-stopping check so that if the
            # process is killed mid-check the checkpoint is still valid.
            save_checkpoint(
                epoch         = epoch,
                model         = self.model,
                optimizer     = optimizer,
                scheduler     = scheduler,
                best_f1       = best_f1,
                best_state    = best_state,
                early_stopper = early_stopper,
                history       = history,
                path          = CHECKPOINT_PATH,
            )
            # ─────────────────────────────────────────────────────────────

            if early_stopper.step(val_metrics["macro_f1"]):
                print(f"\n⏹  Early stopping at epoch {epoch}.\n")
                break

        total_train_time = time.time() - total_train_start
        print(f"\n⏱  Training done: {total_train_time/60:.2f} min — "
              f"{actual_epochs} epochs")

        hist_df = pd.DataFrame(history)
        hist_df.to_csv(os.path.join(OUT_DIR, "history.csv"), index=False)
        self._plot_history(hist_df)

        with open(os.path.join(OUT_DIR, "timing_summary.json"), "w") as f:
            json.dump({
                "total_training_time_s":   total_train_time,
                "total_training_time_min": total_train_time / 60,
                "avg_epoch_time_s":        total_train_time / max(1, actual_epochs),
                "num_epochs_run":          actual_epochs,
            }, f, indent=2)

        if best_state is not None:
            self._save_model_hf(best_state, tokenizer)

        return hist_df, total_train_time

    def evaluate(self, dataset, rare_ids, split_name="dev",
                 measure_inference_time=False):
        self.model.eval()
        loader = DataLoader(dataset, batch_size=2, shuffle=False,
                            collate_fn=collate_rrc)
        all_preds, all_trues = [], []
        n_samples   = len(dataset)
        infer_start = time.time() if measure_inference_time else None

        with torch.no_grad():
            for input_ids, attention_mask, token_type_ids, labels, lengths in loader:
                input_ids      = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                token_type_ids = token_type_ids.to(self.device)
                lengths        = lengths.to(self.device)
                decoded, _ = self.model(input_ids, attention_mask, token_type_ids,
                                        labels=None, lengths=lengths)
                for i, seq_preds in enumerate(decoded):
                    true_len = int(lengths[i].item())
                    true_seq = labels[i, :true_len].cpu().numpy().tolist()
                    all_preds.extend(seq_preds)
                    all_trues.extend(true_seq)

        infer_info = None
        if measure_inference_time:
            total_infer_time = time.time() - infer_start
            n_sentences      = len(all_trues)
            infer_info = {
                "split":                      split_name,
                "n_documents":                n_samples,
                "n_sentences":                n_sentences,
                "total_inference_time_s":     total_infer_time,
                "latency_per_document_ms":    total_infer_time / max(1, n_samples) * 1000,
                "latency_per_sentence_ms":    total_infer_time / max(1, n_sentences) * 1000,
                "throughput_sentences_per_s": n_sentences / max(1e-9, total_infer_time),
            }
            with open(os.path.join(OUT_DIR,
                                   f"inference_time_{split_name}.json"), "w") as f:
                json.dump(infer_info, f, indent=2)
            print(f"\n⏱  Inference ({split_name}): {total_infer_time:.2f}s | "
                  f"throughput: {infer_info['throughput_sentences_per_s']:.1f} sent/s")

        metrics = compute_all_metrics(all_trues, all_preds, rare_ids, split_name)
        if infer_info:
            metrics["inference_time_info"] = infer_info
        return metrics

    def _save_model_hf(self, state_dict, tokenizer):
        self.model.bert.config.save_pretrained(BEST_MODEL_DIR)
        tokenizer.save_pretrained(BEST_MODEL_DIR)
        torch.save(state_dict, os.path.join(BEST_MODEL_DIR, "pytorch_model.bin"))
        model_args = {
            "bert_model_name":    INLEGALBERT_MODEL_NAME,
            "sent_lstm_hidden":   SENT_LSTM_HIDDEN,
            "sent_lstm_layers":   SENT_LSTM_LAYERS,
            "ctx_lstm_hidden":    CTX_LSTM_HIDDEN,
            "ctx_lstm_layers":    CTX_LSTM_LAYERS,
            "mha_heads":          MHA_HEADS,
            "mha_dropout":        MHA_DROPOUT,
            "num_labels":         NUM_LABELS,
            "dropout":            DROPOUT,
            "labels":             LABELS,
            "label2id":           label2id,
            "id2label":           id2label,
            "max_seq_length":     MAX_SEQ_LENGTH,
            "rare_threshold":     RARE_THRESHOLD,
            "freeze_layers":      BERT_FREEZE_LAYERS,
            "ce_weight_strategy": CE_WEIGHT_STRATEGY,
            "ce_weight_cap":      CE_WEIGHT_CAP,
            "aux_ce_weight":      AUX_CE_WEIGHT,
        }
        with open(os.path.join(BEST_MODEL_DIR, "model_args.json"), "w") as f:
            json.dump(model_args, f, indent=2)
        print(f"\n💾 Best model saved → {BEST_MODEL_DIR}/")

    @staticmethod
    def _plot_history(hist_df):
        epochs = hist_df["epoch"].tolist()

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(epochs, hist_df["train_loss"], label="Train Loss", marker="o", markersize=3)
        ax.plot(epochs, hist_df["val_loss"],   label="Val Loss",   marker="s", markersize=3)
        ax.set_title("Training vs Validation Loss (CRF + weighted CE)")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
        ax.legend(); ax.grid(True, alpha=0.3)
        plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_loss_curve.png")
        plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        for col, lbl, ls in [
            ("val_macro_f1",    "Macro-F1",    "-"),
            ("val_micro_f1",    "Micro-F1",    "--"),
            ("val_weighted_f1", "Weighted-F1", "-."),
            ("val_rare_f1",     "Rare-F1",     ":"),
        ]:
            axes[0].plot(epochs, hist_df[col], label=lbl, linestyle=ls, markersize=3)
        axes[0].set_title("Validation F1"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
        axes[1].plot(epochs, hist_df["train_loss"], label="Train", marker="o", markersize=3)
        axes[1].plot(epochs, hist_df["val_loss"],   label="Val",   marker="s", markersize=3)
        axes[1].set_title("Loss Curves"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
        plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_f1_curve.png")
        plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        for col, lbl, ls in [
            ("val_macro_precision",    "Macro-Prec",    "-"),
            ("val_micro_precision",    "Micro-Prec",    "--"),
            ("val_weighted_precision", "Weighted-Prec", "-."),
            ("val_rare_precision",     "Rare-Prec",     ":"),
        ]:
            axes[0].plot(epochs, hist_df[col], label=lbl, linestyle=ls)
        axes[0].set_title("Validation Precision")
        axes[0].legend(); axes[0].grid(True, alpha=0.3)
        for col, lbl, ls in [
            ("val_macro_recall",    "Macro-Rec",    "-"),
            ("val_micro_recall",    "Micro-Rec",    "--"),
            ("val_weighted_recall", "Weighted-Rec", "-."),
            ("val_rare_recall",     "Rare-Rec",     ":"),
        ]:
            axes[1].plot(epochs, hist_df[col], label=lbl, linestyle=ls)
        axes[1].set_title("Validation Recall")
        axes[1].legend(); axes[1].grid(True, alpha=0.3)
        plt.tight_layout()
        p = os.path.join(OUT_DIR, "training_pr_curve.png")
        plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

        if "epoch_train_time_s" in hist_df.columns:
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.bar(epochs, hist_df["epoch_train_time_s"], color="steelblue", alpha=0.8)
            ax.axhline(hist_df["epoch_train_time_s"].mean(), color="red",
                       linestyle="--",
                       label=f"Mean = {hist_df['epoch_train_time_s'].mean():.1f}s")
            ax.set_title("Per-Epoch Training Time")
            ax.set_xlabel("Epoch"); ax.set_ylabel("Time (s)")
            ax.legend(); ax.grid(True, alpha=0.3, axis="y")
            plt.tight_layout()
            p = os.path.join(OUT_DIR, "training_time_curve.png")
            plt.savefig(p, dpi=150); plt.close(); print(f"Saved {p}")

    @staticmethod
    def save_confusion_matrix(cm, split_name, rare_labels=None):
        fig, ax = plt.subplots(figsize=(14, 11))
        sns.heatmap(cm, annot=True, fmt="d",
                    xticklabels=LABELS, yticklabels=LABELS, cmap="Blues", ax=ax)
        if rare_labels:
            for tick in ax.get_xticklabels():
                if tick.get_text() in rare_labels:
                    tick.set_color("red")
            for tick in ax.get_yticklabels():
                if tick.get_text() in rare_labels:
                    tick.set_color("red")
        ax.set_title(f"{split_name.capitalize()} Confusion Matrix"
                     + (f"\n(red = rare ≤ {RARE_THRESHOLD*100:.0f}%)"
                        if rare_labels else ""))
        plt.tight_layout()
        path = os.path.join(OUT_DIR, f"{split_name}_confusion_matrix.png")
        plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")

    @staticmethod
    def save_per_class_f1_chart(per_class_metrics, split_name, rare_labels=None):
        f1s    = [per_class_metrics[l]["f1"] for l in LABELS]
        colors = ["tomato" if (rare_labels and l in rare_labels) else "steelblue"
                  for l in LABELS]
        fig, ax = plt.subplots(figsize=(9, 6))
        bars = ax.barh(LABELS, f1s, color=colors, edgecolor="white")
        ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
        ax.set_xlim(0, 1.12)
        ax.set_xlabel("F1 Score")
        ax.set_title(f"{split_name.capitalize()} Per-Class F1"
                     + (f"\n(red = rare ≤ {RARE_THRESHOLD*100:.0f}%)"
                        if rare_labels else ""))
        ax.grid(True, alpha=0.3, axis="x")
        plt.tight_layout()
        path = os.path.join(OUT_DIR, f"{split_name}_per_class_f1.png")
        plt.savefig(path, dpi=150); plt.close(); print(f"Saved {path}")


# ═══════════════════════════════════════════════════════════
# PRINT FULL METRICS TABLE
# ═══════════════════════════════════════════════════════════
def print_metrics_table(dev_metrics, test_metrics, total_train_time=None,
                        total_trainable=None, total_frozen=None):
    rows = [
        ("Accuracy",           "accuracy"),
        ("Macro-F1",           "macro_f1"),
        ("Micro-F1",           "micro_f1"),
        ("Weighted-F1",        "weighted_f1"),
        ("Rare / Minority F1", "rare_f1"),
        ("Macro-Precision",    "macro_precision"),
        ("Micro-Precision",    "micro_precision"),
        ("Weighted-Precision", "weighted_precision"),
        ("Rare-Precision",     "rare_precision"),
        ("Macro-Recall",       "macro_recall"),
        ("Micro-Recall",       "micro_recall"),
        ("Weighted-Recall",    "weighted_recall"),
        ("Rare-Recall",        "rare_recall"),
    ]
    print("\n" + "=" * 68)
    print("FINAL RESULTS  (InLegalBERT + BiLSTM + MHA + CRF + weighted CE)")
    print("=" * 68)
    if total_trainable is not None:
        print(f"  Trainable Parameters : {total_trainable:,}")
        print(f"  Frozen Parameters    : {total_frozen:,}")
    if total_train_time is not None:
        print(f"  Total Training Time  : {total_train_time/60:.2f} min")
    print(f"  CE Weight Strategy   : {CE_WEIGHT_STRATEGY}  (cap={CE_WEIGHT_CAP},"
          f" aux_weight={AUX_CE_WEIGHT})")
    print("-" * 68)
    print(f"  {'Metric':<28} {'Dev':>12} {'Test':>12}")
    print("-" * 68)
    for label, key in rows:
        if key == "rare_f1":
            print("─" * 68)
        print(f"  {label:<28} {dev_metrics[key]:>12.4f} {test_metrics[key]:>12.4f}")
    print("=" * 68)

    print("\n  PER-CLASS F1 / PRECISION / RECALL")
    print("  " + "-" * 62)
    print(f"  {'Label':<20} {'F1-Dev':>9} {'F1-Test':>9} "
          f"{'Prec-Test':>11} {'Rec-Test':>10}")
    print("  " + "-" * 62)
    for lbl in LABELS:
        dv = dev_metrics["per_class_metrics"][lbl]
        ts = test_metrics["per_class_metrics"][lbl]
        print(f"  {lbl:<20} {dv['f1']:>9.4f} {ts['f1']:>9.4f} "
              f"{ts['precision']:>11.4f} {ts['recall']:>10.4f}")
    print("  " + "-" * 62)


# ═══════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════
def main():
    print(f"Device : {DEVICE}")
    print("Architecture : InLegalBERT (top-4 layers) → "
          "Sentence BiLSTM(128) → MHA(4-head) → "
          "Context BiLSTM(64) → Linear → CRF + class-weighted CE")
    print(f"\nSettings:")
    print(f"  Dropout           : {DROPOUT}")
    print(f"  Weight decay      : {WEIGHT_DECAY}")
    print(f"  BERT freeze       : bottom {BERT_FREEZE_LAYERS} layers")
    print(f"  Aux CE weight     : {AUX_CE_WEIGHT}  (label_smoothing={LABEL_SMOOTHING})")
    print(f"  CE weight strategy: {CE_WEIGHT_STRATEGY}  (cap={CE_WEIGHT_CAP}, norm=max)")
    print(f"  Early stopping    : patience={ES_PATIENCE}")
    print(f"  Checkpoint dir    : {CHECKPOINT_DIR}\n")   # [CKPT-5]

    train_raw  = load_jsonl(TRAIN_PATH)
    dev_raw    = load_jsonl(DEV_PATH)
    test_raw   = load_jsonl(TEST_PATH)
    train_docs = extract_docs(train_raw)
    dev_docs   = extract_docs(dev_raw)
    test_docs  = extract_docs(test_raw)
    print(f"  Train: {len(train_docs)} | Dev: {len(dev_docs)} | Test: {len(test_docs)}")

    rare_labels, rare_ids, label_freqs = detect_rare_classes(train_docs)
    pd.DataFrame([
        {"label": l, "frequency": label_freqs[l], "is_rare": l in rare_labels}
        for l in LABELS
    ]).to_csv(os.path.join(OUT_DIR, "label_frequencies.csv"), index=False)

    class_weights = compute_class_weights(
        train_docs,
        strategy = CE_WEIGHT_STRATEGY,
        cap      = CE_WEIGHT_CAP,
        device   = DEVICE,
    )
    weights_df = pd.DataFrame({
        "label":  LABELS,
        "weight": class_weights.cpu().numpy(),
    })
    weights_df.to_csv(os.path.join(OUT_DIR, "ce_class_weights.csv"), index=False)
    print(f"  CE class weights saved → {OUT_DIR}/ce_class_weights.csv")

    print("Loading tokenizer...")
    tokenizer     = AutoTokenizer.from_pretrained(INLEGALBERT_MODEL_NAME)
    train_dataset = RRCDataset(train_docs, tokenizer)
    dev_dataset   = RRCDataset(dev_docs,   tokenizer)
    test_dataset  = RRCDataset(test_docs,  tokenizer)

    print("\nInitialising model...")
    model = InLegalBERT_BiLSTM_MHA_CRF(
        bert_model_name  = INLEGALBERT_MODEL_NAME,
        sent_lstm_hidden = SENT_LSTM_HIDDEN,
        sent_lstm_layers = SENT_LSTM_LAYERS,
        ctx_lstm_hidden  = CTX_LSTM_HIDDEN,
        ctx_lstm_layers  = CTX_LSTM_LAYERS,
        mha_heads        = MHA_HEADS,
        mha_dropout      = MHA_DROPOUT,
        num_labels       = NUM_LABELS,
        dropout          = DROPOUT,
        freeze_layers    = BERT_FREEZE_LAYERS,
        class_weights    = class_weights,
    )

    total_trainable, total_frozen, param_table = count_parameters(model)
    pd.DataFrame(param_table).to_csv(
        os.path.join(OUT_DIR, "parameter_summary.csv"), index=False
    )

    trainer = Trainer(model, device=DEVICE)
    print(f"\nStarting training (max {NUM_EPOCHS} epochs, "
          f"ES patience={ES_PATIENCE})...")
    hist_df, total_train_time = trainer.train(
        train_dataset, dev_dataset,
        rare_ids   = rare_ids,
        tokenizer  = tokenizer,
        num_epochs = NUM_EPOCHS,
    )
    print("\nTraining complete.")

    best_bin = os.path.join(BEST_MODEL_DIR, "pytorch_model.bin")
    if os.path.exists(best_bin):
        model.load_state_dict(torch.load(best_bin, map_location=DEVICE))
        print("Loaded best checkpoint.")

    print("\nEvaluating on Dev set...")
    dev_metrics = trainer.evaluate(dev_dataset, rare_ids, split_name="dev",
                                   measure_inference_time=True)
    print(f"  Dev  Accuracy : {dev_metrics['accuracy']:.4f}")
    print(f"  Dev  Macro-F1 : {dev_metrics['macro_f1']:.4f}")
    print(f"  Dev  Rare-F1  : {dev_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "dev_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT + BiLSTM + MHA + CRF + class-weighted CE\n")
        f.write(f"CE weight strategy: {CE_WEIGHT_STRATEGY}"
                f" (cap={CE_WEIGHT_CAP}, aux_weight={AUX_CE_WEIGHT}, norm=max)\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(dev_metrics["cls_report"])

    trainer.save_confusion_matrix(dev_metrics["cm"], "dev", rare_labels)
    trainer.save_per_class_f1_chart(dev_metrics["per_class_metrics"], "dev", rare_labels)

    print("\nEvaluating on Test set...")
    test_metrics = trainer.evaluate(test_dataset, rare_ids, split_name="test",
                                    measure_inference_time=True)
    print(f"  Test Accuracy : {test_metrics['accuracy']:.4f}")
    print(f"  Test Macro-F1 : {test_metrics['macro_f1']:.4f}")
    print(f"  Test Rare-F1  : {test_metrics['rare_f1']:.4f}")

    with open(os.path.join(OUT_DIR, "test_classification_report.txt"), "w") as f:
        f.write("Model: InLegalBERT + BiLSTM + MHA + CRF + class-weighted CE\n")
        f.write(f"CE weight strategy: {CE_WEIGHT_STRATEGY}"
                f" (cap={CE_WEIGHT_CAP}, aux_weight={AUX_CE_WEIGHT}, norm=max)\n")
        f.write(f"Rare classes (≤{RARE_THRESHOLD*100:.0f}%): {rare_labels}\n\n")
        f.write(test_metrics["cls_report"])

    trainer.save_confusion_matrix(test_metrics["cm"], "test", rare_labels)
    trainer.save_per_class_f1_chart(test_metrics["per_class_metrics"], "test", rare_labels)

    pd.DataFrame({
        "true": [id2label[x] for x in test_metrics["all_trues"]],
        "pred": [id2label[x] for x in test_metrics["all_preds"]],
    }).to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

    for split, mets in [("dev", dev_metrics), ("test", test_metrics)]:
        pd.DataFrame([
            {
                "label":     lbl,
                "is_rare":   lbl in rare_labels,
                "f1":        mets["per_class_metrics"][lbl]["f1"],
                "precision": mets["per_class_metrics"][lbl]["precision"],
                "recall":    mets["per_class_metrics"][lbl]["recall"],
            }
            for lbl in LABELS
        ]).to_csv(os.path.join(OUT_DIR, f"{split}_per_class_metrics.csv"), index=False)

    scalar_keys = [
        "macro_f1","micro_f1","weighted_f1","rare_f1",
        "macro_precision","micro_precision","weighted_precision","rare_precision",
        "macro_recall","micro_recall","weighted_recall","rare_recall","accuracy",
    ]
    with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
        json.dump({
            "model": {
                "name":               "InLegalBERT + BiLSTM + MHA + CRF + weighted CE",
                "bert_model":         INLEGALBERT_MODEL_NAME,
                "sent_lstm_hidden":   SENT_LSTM_HIDDEN,
                "ctx_lstm_hidden":    CTX_LSTM_HIDDEN,
                "mha_heads":          MHA_HEADS,
                "trainable_params":   total_trainable,
                "frozen_params":      total_frozen,
                "ce_weight_strategy": CE_WEIGHT_STRATEGY,
                "ce_weight_cap":      CE_WEIGHT_CAP,
                "aux_ce_weight":      AUX_CE_WEIGHT,
                "ce_weight_norm":     "max",
                "class_weights":      {
                    id2label[i]: float(class_weights[i]) for i in range(NUM_LABELS)
                },
            },
            "timing": {
                "total_training_time_s":   total_train_time,
                "total_training_time_min": total_train_time / 60,
                "dev_inference":  dev_metrics.get("inference_time_info", {}),
                "test_inference": test_metrics.get("inference_time_info", {}),
            },
            "rare_classes":   rare_labels,
            "dev":  {k: dev_metrics[k]  for k in scalar_keys},
            "test": {k: test_metrics[k] for k in scalar_keys},
        }, f, indent=2)

    print_metrics_table(
        dev_metrics, test_metrics,
        total_train_time = total_train_time,
        total_trainable  = total_trainable,
        total_frozen     = total_frozen,
    )
    print(f"\n📁 All outputs → {OUT_DIR}/")


if __name__ == "__main__":
    main()